# Site Reliability Server — GRPO Training Pipeline

**What this notebook does:**  
Trains a small LLM to act as an on-call SRE agent inside a production-style incident response environment using GRPO reinforcement learning (TRL + HuggingFace PEFT). The agent learns to diagnose failures, issue remediations, and recover service health across 5 progressive tasks.

**Estimated runtime:** ~20–25 minutes on a free Colab T4  
**GPU requirement:** T4 (minimum) — Runtime → Change runtime type → T4 GPU  
**Repo:** https://github.com/PreetKot/Site-Reliability-Server

---
**Cell execution order:** Run all cells top-to-bottom. No manual input required.

## Step 1 — Clone the repository

In [ ]:
import os

REPO_URL = "https://github.com/PreetKot/Site-Reliability-Server.git"
REPO_DIR = "Site-Reliability-Server"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL}
else:
    print(f"Repo already cloned at {REPO_DIR}, pulling latest...")
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
print("Working directory:", os.getcwd())

## Step 2 — Install all dependencies (server + RL stack)

In [ ]:
# Install server runtime dependencies first
!pip install -q -r requirements.txt

# Install RL training stack (TRL + HuggingFace PEFT + bitsandbytes)
# No Unsloth — avoids TRL/Unsloth version-mismatch errors on Colab
!pip install -q -r requirements_rl.txt

print("All dependencies installed.")

## Step 3 — Generate scenario files for all 5 tasks

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, "env/data_generator.py"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("Scenario generation failed")

# Quick sanity check: list generated scenario counts per task
from pathlib import Path
for task in ["easy", "medium", "hard", "expert", "enterprise"]:
    count = len(list(Path(f"scenarios/{task}").glob("*.json")))
    print(f"  {task}: {count} scenarios")

## Step 4 — Disable W&B telemetry (optional: remove to enable W&B tracking)

In [ ]:
import os

# Telemetry is disabled so the notebook runs end-to-end without interactive prompts.
# To enable W&B tracking: remove the line below and call wandb.login() instead.
os.environ["WANDB_MODE"] = "disabled"

print("W&B telemetry disabled. Training will proceed without cloud logging.")
print("To enable: remove os.environ['WANDB_MODE'] = 'disabled' and call wandb.login()")

## Step 5 — Start the FastAPI OpenEnv server in the background

In [ ]:
import subprocess, time, requests, sys

log_file = open("server.log", "w")
server_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "7860"],
    stdout=log_file,
    stderr=log_file,
)
print(f"Server PID: {server_process.pid}")

# Retry loop — up to 10 attempts, 2s apart (handles slow Colab cold starts)
for attempt in range(1, 11):
    time.sleep(2)
    try:
        res = requests.get("http://localhost:7860/health", timeout=3)
        if res.status_code == 200:
            print(f"Server is RUNNING (attempt {attempt}/10)")
            break
    except requests.exceptions.ConnectionError:
        print(f"  Waiting for server... attempt {attempt}/10")
else:
    print("Server failed to start. Printing server.log:")
    print(open("server.log").read())
    raise RuntimeError("Server did not start within 20 seconds")

## Step 6 — Run GRPO training (3 epochs, Qwen2.5-1.5B-Instruct)

In [ ]:
import subprocess, sys

train_cmd = [
    sys.executable, "train_grpo.py",
    "--epochs", "3",
    "--model_name", "Qwen/Qwen2.5-1.5B-Instruct",
]

print("Starting GRPO training...")
print("Command:", " ".join(train_cmd))
print("-" * 60)

last_lines = []
proc = subprocess.Popen(train_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in iter(proc.stdout.readline, ""):
    print(line, end="", flush=True)
    last_lines.append(line.strip())
    if len(last_lines) > 20:
        last_lines.pop(0)

proc.wait()
print("-" * 60)
if proc.returncode == 0:
    print("Training completed successfully.")
else:
    print(f"Training exited with code {proc.returncode}")

reward_lines = [l for l in last_lines if "reward" in l.lower()]
if reward_lines:
    print("\nFinal reward log line:", reward_lines[-1])
print("Command:", " ".join(train_cmd))
print("-" * 60)

last_lines = []
proc = subprocess.Popen(train_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in iter(proc.stdout.readline, ""):
    print(line, end="", flush=True)
    last_lines.append(line.strip())
    if len(last_lines) > 20:
        last_lines.pop(0)

proc.wait()
print("-" * 60)
if proc.returncode == 0:
    print("Training completed successfully.")
else:
    print(f"Training exited with code {proc.returncode}")

# Extract and display the final reward line
reward_lines = [l for l in last_lines if "reward" in l.lower()]
if reward_lines:
    print("\nFinal reward log line:", reward_lines[-1])

## Step 7 — Display training curves (reward and loss)

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

reward_path = Path("reward_curve.png")
loss_path = Path("loss_curve.png")

missing = [p for p in [reward_path, loss_path] if not p.exists()]
if missing:
    print(f"Warning: missing plot files: {missing}")
    print("Training may not have completed or curves were saved to a different path.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].imshow(mpimg.imread(str(reward_path)))
    axes[0].set_title("Reward Curve", fontsize=14)
    axes[0].axis("off")

    axes[1].imshow(mpimg.imread(str(loss_path)))
    axes[1].set_title("Loss Curve", fontsize=14)
    axes[1].axis("off")

    plt.suptitle("GRPO Training Evidence — Site Reliability Server", fontsize=16)
    plt.tight_layout()
    plt.savefig("training_summary.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Training curves displayed and saved to training_summary.png")

## Step 8 — Baseline sanity check (environment works end-to-end)

In [ ]:
# Demonstrates the full reset → step → grader loop against the running server.
# This is NOT inference from the trained model — it is a structural smoke test
# that confirms the environment API is healthy and scoring works correctly.

import requests, json

BASE = "http://localhost:7860"

# 1. Reset episode
reset_resp = requests.post(f"{BASE}/reset", json={"task_id": "easy"})
assert reset_resp.status_code == 200, f"Reset failed: {reset_resp.text}"
obs = reset_resp.json()
print(f"Reset OK | task={obs['task_id']} | step={obs['step']}/{obs['max_steps']}")
print(f"  Active alerts: {len(obs.get('active_alerts', []))}")
print(f"  Health overall: {obs['health_summary']['overall']:.3f}")

# 2. Send 3 diagnostic steps (CHECK_LOGS is always a safe starting action)
DEMO_ACTIONS = [
    {"action_type": "CHECK_LOGS",      "target_service": "api-gateway",  "reason": "checking for error patterns"},
    {"action_type": "INSPECT_SERVICE", "target_service": "auth-service", "reason": "inspecting upstream dependency"},
    {"action_type": "RESTART_SERVICE", "target_service": "api-gateway",  "reason": "applying remediation after diagnosis"},
]

cumulative = 0.0
for i, action in enumerate(DEMO_ACTIONS, 1):
    step_resp = requests.post(f"{BASE}/step", json=action)
    assert step_resp.status_code == 200, f"Step {i} failed: {step_resp.text}"
    data = step_resp.json()
    reward = data["reward"]["step_reward"]
    cumulative = data["reward"]["cumulative"]
    health = data["observation"]["health_summary"]["overall"]
    print(f"  Step {i}: action={action['action_type']} | reward={reward:+.4f} | health={health:.3f}")

print(f"\nCumulative reward after 3 steps: {cumulative:+.4f}")

# 3. Call grader
state_resp = requests.get(f"{BASE}/state")
grader_resp = requests.post(f"{BASE}/grader", json=state_resp.json())
assert grader_resp.status_code == 200, f"Grader failed: {grader_resp.text}"
grade = grader_resp.json()
print(f"\nGrader score: {grade['score']:.4f}")
print("Breakdown:", json.dumps(grade.get('breakdown', {}), indent=2))

print("\nSanity check PASSED — environment reset/step/grader loop is fully operational.")